# Token Dynamics Landscape

Map how token representations evolve through the model:
velocity, curvature, convergence, and drift from a reference layer.

## Why This Matters

Token dynamics landscape tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_dynamics_landscape import (
    token_velocity, token_curvature, token_convergence,
    token_representation_drift, token_dynamics_landscape_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Token Velocity

How fast does each token's representation change per layer?

In [ ]:
result = token_velocity(model, tokens)
print(f"Fastest layer: {result['fastest_layer']}")
for p in result['per_layer']:
    bar = '█' * int(p['mean_velocity'] * 10)
    print(f"  Layer {p['layer']}: mean={p['mean_velocity']:.4f} "
          f"max={p['max_velocity']:.4f} {bar}")

## Curvature

How much does the direction of change shift between layers?

In [ ]:
result = token_curvature(model, tokens)
print(f"Mean curvature: {result['mean_curvature']:.4f}")
for p in result['per_transition']:
    bar = '█' * int(p['curvature'] * 20)
    print(f"  L{p['from_layer']}->L{p['to_layer']}: "
          f"cos={p['mean_cosine']:.4f} curv={p['curvature']:.4f} {bar}")

## Token Convergence

Do different token positions converge or diverge through layers?

In [ ]:
result = token_convergence(model, tokens)
print(f"Trend: {result['trend']}")
for p in result['per_layer']:
    bar = '█' * int((p['mean_pairwise_similarity'] + 1) * 15)
    print(f"  Layer {p['layer']}: similarity={p['mean_pairwise_similarity']:.4f} {bar}")

## Representation Drift

How far has each token drifted from its layer-0 representation?

In [ ]:
result = token_representation_drift(model, tokens, reference_layer=0)
for p in result['per_layer']:
    bar = '█' * int(p['mean_drift'] * 20)
    print(f"  Layer {p['layer']}: mean_drift={p['mean_drift']:.4f} {bar}")

## Dynamics Summary

In [ ]:
result = token_dynamics_landscape_summary(model, tokens)
print(f"Fastest layer: {result['fastest_layer']}")
print(f"Mean curvature: {result['mean_curvature']:.4f}")
print(f"Convergence trend: {result['convergence_trend']}")